# Phase 2, Step 0 — move to GPU and fix the speed

Phase 1 is done: **0.5965 ± 0.0103** on the full train split.

The problem: 1,468 seconds per seed means 2 hours per configuration. Phase 2 has
four pieces, each needing an ablation table and a hyperparameter sweep. That is
dozens of configurations. At 2 hours each it is a month of waiting.

This notebook does two things:
1. **Benchmark** batch size and CRF on/off to find the fastest setting
2. **Verify** the GPU reproduces the laptop's numbers before we trust it

The second one is not optional. If the GPU quietly gives different answers, every
Phase 2 result would be incomparable with Phase 1.

---
**Runtime → Change runtime type → T4 GPU → Save**


## 1. GPU check


In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'


## 2. Install


In [ ]:
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
from torchcrf import CRF
print('ok')


## 3. Get the code

**Edit REPO below.** Push your Phase 1 code to GitHub first.


In [ ]:
REPO = 'https://github.com/YOUR-USERNAME/YOUR-REPO.git'   # <-- EDIT

import os, shutil
if os.path.exists('project'): shutil.rmtree('project')
!git clone -q $REPO project
%cd project
!ls src/ notebooks/


*Private repo? Skip the cell above, zip your `src/` and `notebooks/` folders, and use this instead:*


In [ ]:
# from google.colab import files
# import zipfile, os
# up = files.upload()
# os.makedirs('project', exist_ok=True)
# zipfile.ZipFile(list(up)[0]).extractall('project')
# %cd project
# !ls


## 4. Download the fastText vectors

About 600 MB. Two to three minutes on Colab's connection.


In [ ]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q --show-progress -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!ls -lh embeddings/


## 5. Rebuild the embedding artifacts

Same as Step 3. Confirm the printed numbers match your laptop run:
vocabulary **28,456**, coverage **81.9%**, test unseen tokens **14.3%**.


In [ ]:
!python notebooks/03_embeddings.py 2>&1 | grep -E 'vocab|coverage|found|missing|unseen|dimension|shape'


## 6. SPEED BENCHMARK

Times **one epoch** for six settings. One epoch is enough to compare speed and
takes minutes instead of hours.

- **batch size** 32 / 64 / 128 — bigger batches mean fewer weight updates per
  epoch, usually faster on a GPU
- **CRF on / off** — CRF decoding is sequential and may be the bottleneck

Your laptop baseline for comparison: about **49 seconds per epoch** (1,468s ÷ 30).


In [ ]:
import sys, time
sys.path.insert(0, 'src')
import numpy as np, torch

from data import load_sold, train_val_split
from embeddings import build_vocab
from dataset import make_loader
from model import BiLSTMTagger
from train import set_seed, evaluate, get_device

device = get_device()
train_full = load_sold('train')
train_part, val = train_val_split(train_full)
vocab, _ = build_vocab(train_part['token_list'], min_freq=1)
matrix = np.load('artifacts/embedding_matrix.npy')
print(f'device {device} | train-part {len(train_part):,} | vocab {len(vocab):,}')


In [ ]:
def time_epoch(batch_size, use_crf, seed=1):
    g = set_seed(seed)
    m = BiLSTMTagger(matrix, hidden_size=64, dropout=0.5,
                     freeze_embeddings=True, use_crf=use_crf).to(device)
    tr = make_loader(train_part, vocab, batch_size, shuffle=True, generator=g)
    va = make_loader(val, vocab, batch_size, shuffle=False)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)

    torch.cuda.synchronize(); t0 = time.time()
    m.train()
    for ids, labels, mask, lengths, _ in tr:
        ids, labels, mask = ids.to(device), labels.to(device), mask.to(device)
        opt.zero_grad()
        loss = m.loss(ids, labels, mask, lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        opt.step()
    torch.cuda.synchronize(); train_s = time.time() - t0

    t1 = time.time(); _ = evaluate(m, va, device); eval_s = time.time() - t1
    return train_s, eval_s

print(f"{'batch':>6} {'CRF':>6} {'train':>9} {'eval':>8} {'per epoch':>11} {'5 seeds x 30ep':>16} {'vs laptop':>10}")
print('-' * 74)
LAPTOP = 49.0
rows = []
for bs in [32, 64, 128]:
    for crf in [True, False]:
        tr_s, ev_s = time_epoch(bs, crf)
        tot = tr_s + ev_s
        rows.append((bs, crf, tot))
        print(f'{bs:>6} {str(crf):>6} {tr_s:>8.1f}s {ev_s:>7.1f}s {tot:>10.1f}s'
              f' {tot*30*5/60:>13.0f} min {LAPTOP/tot:>9.1f}x')

best_crf = min([r for r in rows if r[1]], key=lambda r: r[2])
print(f'\nFastest WITH CRF: batch_size={best_crf[0]} at {best_crf[2]:.1f}s/epoch')
print('Keep the CRF for reported results. Only drop it for sweeps if it dominates.')


## 7. VERIFICATION — does the GPU agree with the laptop?

Runs seed 1 with the exact Phase 1 settings (batch 32, CRF on, 60 epochs,
patience 12) and compares against the laptop's seed 1 result of **0.6023**.

**Expect close, not identical.** Different hardware uses different floating-point
kernels, so the same seed does not give bit-identical results across devices.
Within about ±0.02 is fine. A gap of 0.05 or more means something is wrong and
must be found before Phase 2 starts.

Takes a few minutes.


In [ ]:
!python notebooks/04_baseline.py --seeds 1 --epochs 60 --patience 12 \
    --batch-size 32 --tag colab_verify 2>&1 | tail -12


In [ ]:
LAPTOP_SEED1 = 0.6023
gpu = float(input('Enter the TEST F1 printed above: '))
d = abs(gpu - LAPTOP_SEED1)
print(f'\nlaptop seed 1 : {LAPTOP_SEED1:.4f}')
print(f'colab  seed 1 : {gpu:.4f}')
print(f'difference    : {d:.4f}')
if d <= 0.02:
    print('\nOK. Hardware difference only. Safe to run Phase 2 on Colab.')
elif d <= 0.05:
    print('\nBORDERLINE. Run seeds 2 and 3 too before trusting it.')
else:
    print('\nTOO BIG. Do not proceed. Check: same vocab size? same embedding')
    print('coverage? same split seed? same package versions?')


## 8. Batch-size score check

Speed is not the only thing batch size changes. Bigger batches mean fewer weight
updates per epoch, which can change the **score** too.

This runs the fastest batch size on 3 seeds and compares against batch 32. If the
score holds, adopt it for Phase 2. If it drops meaningfully, stay at 32 even
though it is slower — speed is a means, not the goal.


In [ ]:
BS = 64   # <-- set to the fastest batch size from cell 6

!python notebooks/04_baseline.py --seeds 1 2 3 --epochs 60 --patience 12 \
    --batch-size $BS --tag colab_bs_check 2>&1 | tail -14


## 9. Save results

Colab wipes everything when the session ends.


In [ ]:
from google.colab import files
import os
if os.path.exists('results/results.csv'):
    files.download('results/results.csv')
else:
    print('no results.csv')


---
## Record in the README before moving on

1. **The speed table from cell 6.** Batch size, CRF on/off, seconds per epoch,
   speedup over the laptop. These also feed the paper's computational analysis.
2. **The verification result from cell 7.** State plainly that Colab reproduced
   the laptop baseline within X, so Phase 2 numbers are comparable to Phase 1.
3. **The batch-size decision from cell 8**, and the score it gave.

Then Piece 1 begins.
